# Package

In [1]:
import subprocess
import sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

try:
    import pymoo
except ImportError:
    pip_install("pymoo")

try:
    import torch
except ImportError:
    pip_install("torch")

try:
    import sklearn
except ImportError:
    pip_install("scikit-learn")

In [2]:
import numpy as np
import time

from pymoo.problems import get_problem
from pymoo.operators.sampling.lhs import LHS
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.optimize import minimize
from pymoo.core.problem import Problem
from pymoo.problems.multi.omnitest import OmniTest

from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans
from pymoo.core.problem import Problem
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting
import plotly.graph_objects as go

from scipy.spatial.distance import cdist

import torch
import torch.nn as nn
import torch.optim as optim

# Metrics
from pymoo.indicators.hv import HV
from pymoo.indicators.igd_plus import IGDPlus
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Class

In [3]:
# Plot: 2 Objs and pareto front
def plot_obj_2d(F, xlim=(0, 1), ylim=(0, 1)):
    n_obj = F.shape[1]
    if n_obj == 2:
        nds = NonDominatedSorting()
        front_idx = nds.do(F, only_non_dominated_front=True)

        pareto_F = F[front_idx]
        non_pareto_F = np.delete(F, front_idx, axis=0)

        fig = go.Figure(
            data=go.Scatter(
                x=F[:, 0],
                y=F[:, 1],
                mode='markers',
                name='Objective Values',
                marker=dict(size=6, color='#87CEEB', opacity=0.7)
            )
        )
        fig.add_trace(go.Scatter(
            x=pareto_F[:, 0],
            y=pareto_F[:, 1],
            mode='markers',
            name='Pareto Front',
            marker=dict(size=7, color='#FF7F0E', opacity=0.9, symbol='diamond')
    ))
        fig.update_layout(
            xaxis_title='f1',
            yaxis_title='f2',
            width=600,
            height=600,
            xaxis=dict(range=list(xlim)),
            yaxis=dict(range=list(ylim))
        )
    fig.show()


# Metrics: PICP
def metrics_picp(y_true, y_lower, y_upper):
    y_true = np.asarray(y_true)
    y_lower = np.asarray(y_lower)
    y_upper = np.asarray(y_upper)

    inside = (y_true >= y_lower) & (y_true <= y_upper)
    picp_per_objective = np.mean(inside, axis=0)
    picp_mean = np.mean(picp_per_objective)

    return picp_per_objective, picp_mean

def mean_std(arr):
    return np.mean(arr), np.std(arr)

In [4]:
class RBFN:
    def __init__(self, n_centers=None, gamma=0.5, lambda_reg=1e-6, random_state=0):
        self.n_centers = n_centers  # if None -> use min(N, D)
        self.gamma = gamma
        self.lambda_reg = lambda_reg
        self.random_state = random_state

        self.centers = None
        self.weights = None

    def _rbf(self, r):
        return np.exp(-self.gamma * r * r)

    def fit(self, X, y):
        """
        X: (N, D)
        y: (N, 1)
        """
        X = np.asarray(X)
        y = np.asarray(y).reshape(-1, 1)
        N, D = X.shape

        if self.n_centers is None:
            n_centers = D
        else:
            n_centers = self.n_centers

        n_centers = min(N, n_centers)

        kmeans = KMeans(n_clusters=n_centers, random_state=self.random_state, n_init=5)
        kmeans.fit(X)
        self.centers = kmeans.cluster_centers_

        dist = cdist(X, self.centers)
        Phi = self._rbf(dist)

        A = Phi.T @ Phi + self.lambda_reg * np.eye(n_centers)
        b = Phi.T @ y
        self.weights = np.linalg.solve(A, b)

    def predict(self, X_new):
        X_new = np.asarray(X_new)
        dist = cdist(X_new, self.centers)
        Phi_new = self._rbf(dist)
        mu = Phi_new @ self.weights
        std = np.zeros_like(mu)
        return mu, std


class SurrogateRBFN:

    def __init__(self, gamma=0.5, lambda_reg=1e-6, random_state=0):
        self.model = None
        self.gamma = gamma
        self.lambda_reg = lambda_reg
        self.random_state = random_state

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y).reshape(-1, 1)
        N, D = X.shape
        rbfn = RBFN(n_centers=D, gamma=self.gamma,
                    lambda_reg=self.lambda_reg,
                    random_state=self.random_state)
        rbfn.fit(X, y)
        self.model = rbfn

    def predict(self, X):
        return self.model.predict(X)



class Generator(nn.Module):
    def __init__(self, z_dim, D, M):
        super().__init__()
        out_dim = D + M
        self.net = nn.Sequential(
            nn.Linear(z_dim, D),
            nn.ReLU(),
            nn.Linear(D, D),
            nn.ReLU(),
            nn.Linear(D, out_dim),
            nn.Tanh()
        )

    def forward(self, z):
        return self.net(z)


class Discriminator(nn.Module):
    def __init__(self, D, M):
        super().__init__()
        in_dim = D + M
        self.net = nn.Sequential(
            nn.Linear(in_dim, D),
            nn.ReLU(),
            nn.Linear(D, D),
            nn.ReLU(),
            nn.Linear(D, 1)
        )

    def forward(self, x):
        return self.net(x)


def gradient_penalty(D_net, real_samples, fake_samples, device):
    alpha = torch.rand(real_samples.size(0), 1, device=device)
    alpha = alpha.expand_as(real_samples)
    interpolates = alpha * real_samples + ((1 - alpha) * fake_samples)
    interpolates.requires_grad_(True)

    d_interpolates = D_net(interpolates)
    fake = torch.ones(real_samples.size(0), 1, device=device)

    gradients = torch.autograd.grad(
        outputs=d_interpolates,
        inputs=interpolates,
        grad_outputs=fake,
        create_graph=True,
        retain_graph=True,
        only_inputs=True
    )[0]

    gradients = gradients.view(gradients.size(0), -1)
    grad_norm = gradients.norm(2, dim=1)
    gp = ((grad_norm - 1) ** 2).mean()
    return gp


def train_wgan_gp(joint_init,
                  D_dim,
                  n_epochs=2000,
                  batch_size=64,
                  z_dim=32,
                  lambda_gp=10.0,
                  n_critic=5,
                  lr=1e-4,
                  verbose=True):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    G = Generator(z_dim, D_dim - 2, 2).to(device)
    D_net = Discriminator(D_dim - 2, 2).to(device)

    optimizer_G = optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.9))
    optimizer_D = optim.Adam(D_net.parameters(), lr=lr, betas=(0.5, 0.9))

    joint_t = torch.tensor(joint_init, dtype=torch.float32, device=device)
    N = joint_init.shape[0]
    batch_size = min(batch_size, N)

    if verbose:
        print("Training WGAN-GP...")

    for epoch in range(n_epochs):

        idx = np.random.randint(0, N, batch_size)
        real_batch = joint_t[idx]

        for _ in range(n_critic):
            z = torch.randn(batch_size, z_dim, device=device)
            fake_batch = G(z)

            optimizer_D.zero_grad()
            real_validity = D_net(real_batch)
            fake_validity = D_net(fake_batch.detach())
            gp = gradient_penalty(D_net, real_batch, fake_batch, device)
            d_loss = -torch.mean(real_validity) + torch.mean(fake_validity) + lambda_gp * gp
            d_loss.backward()
            optimizer_D.step()

        z = torch.randn(batch_size, z_dim, device=device)
        optimizer_G.zero_grad()
        fake_batch = G(z)
        fake_validity = D_net(fake_batch)
        g_loss = -torch.mean(fake_validity)
        g_loss.backward()
        optimizer_G.step()

        if verbose and (epoch + 1) % 500 == 0:
            print(f"Epoch {epoch+1}/{n_epochs} | D_loss: {d_loss.item():.4f} | G_loss: {g_loss.item():.4f}")

    if verbose:
        print("WGAN-GP training done.\n")

    return G, D_net, device



def build_poly_models(X, F, degree=2):
    polys = []
    regs = []
    for i in range(F.shape[1]):
        poly = PolynomialFeatures(degree=degree, include_bias=True)
        X_poly = poly.fit_transform(X)
        reg = LinearRegression().fit(X_poly, F[:, i])
        polys.append(poly)
        regs.append(reg)
    return polys, regs


def poly_predict(polys, regs, X_new):
    preds = []
    for poly, reg in zip(polys, regs):
        X_poly = poly.transform(X_new)
        y_pred = reg.predict(X_poly)
        preds.append(y_pred.reshape(-1, 1))
    return np.hstack(preds)

def construct_surrogate_pool_with_GAN(
    X_init,
    F_init,
    G,
    D_net,
    device,
    n_models,
    select_ratio=0.2,
    poly_degree=2,
    gamma_rbfn=0.5,
    lambda_rbfn=1e-6
):

    N, D = X_init.shape
    M = F_init.shape[1]

    print("Building surrogate pool with GAN augmentation ...")

    polys, regs = build_poly_models(X_init, F_init, degree=poly_degree)

    X_min = X_init.min(axis=0)
    X_max = X_init.max(axis=0)
    F_min = F_init.min(axis=0)
    F_max = F_init.max(axis=0)

    def norm_joint(x, f):
        xn = 2.0 * (x - X_min) / (X_max - X_min + 1e-12) - 1.0
        fn = 2.0 * (f - F_min) / (F_max - F_min + 1e-12) - 1.0
        return np.hstack([xn, fn])

    def denorm_x(xn):
        return (xn + 1.0) * 0.5 * (X_max - X_min + 1e-12) + X_min

    P_models = [[] for _ in range(M)]

    for k in range(n_models):
        z = torch.randn(N, G.net[0].in_features, device=device)
        with torch.no_grad():
            joint_syn = G(z).cpu().numpy()

        X_syn_n = joint_syn[:, :D]
        X_syn = denorm_x(X_syn_n)

        with torch.no_grad():
            scores = D_net(torch.tensor(joint_syn, dtype=torch.float32, device=device)).cpu().numpy().flatten()

        idx_sorted = np.argsort(-scores)
        top_n = max(1, int(select_ratio * N))
        idx_top = idx_sorted[:top_n]
        X_h = X_syn[idx_top]

        F_h = poly_predict(polys, regs, X_h)

        X_train = np.vstack([X_init, X_h])
        for j in range(M):
            y_train = np.vstack([F_init[:, j:j+1], F_h[:, j:j+1]])
            model = SurrogateRBFN(gamma=gamma_rbfn, lambda_reg=lambda_rbfn, random_state=k)
            model.fit(X_train, y_train)
            P_models[j].append(model)

        print(f"  Surrogate model {k+1}/{n_models} built. Train size = {X_train.shape[0]}")

    print("Surrogate pool ready.\n")
    return P_models, (X_min, X_max, F_min, F_max)



def surrogate_predict_with_ensemble(x, surrogate_pools):
    x = np.asarray(x)
    N = x.shape[0]
    M = len(surrogate_pools)
    y_mean = np.zeros((N, M))

    for m in range(M):
        preds_m = []
        for model in surrogate_pools[m]:
            mu, _ = model.predict(x)
            preds_m.append(mu)
        preds_m = np.hstack(preds_m)  # (N, K)
        y_mean[:, m:m+1] = preds_m.mean(axis=1, keepdims=True)

    return y_mean


def discriminator_confidence_score(x, y_pred, D_net, device, X_min, X_max, F_min, F_max):
    Xn = 2.0 * (x - X_min) / (X_max - X_min + 1e-12) - 1.0
    Yn = 2.0 * (y_pred - F_min) / (F_max - F_min + 1e-12) - 1.0
    joint = np.hstack([Xn, Yn])
    joint_t = torch.tensor(joint, dtype=torch.float32, device=device)
    with torch.no_grad():
        logits = D_net(joint_t).cpu().numpy().flatten()
    cscore = 1.0 / (1.0 + np.exp(-logits))
    return cscore.reshape(-1, 1)


def critical_fitness(x,
                     surrogate_pools,
                     D_net,
                     device,
                     X_min, X_max, F_min, F_max,
                     alpha_critic=0.1):
    """
    fitness_critic(x) = (1/K Σ y_i) * (1 - α * Cscore)
    """
    y_mean = surrogate_predict_with_ensemble(x, surrogate_pools)
    cscore = discriminator_confidence_score(x, y_mean, D_net, device, X_min, X_max, F_min, F_max)
    factor = (1.0 - alpha_critic * cscore)
    return y_mean * factor


class DDMOEA_GAN_Problem(Problem):
    def __init__(self,
                 n_var,
                 n_obj,
                 xl,
                 xu,
                 surrogate_pools,
                 D_net,
                 device,
                 X_min, X_max, F_min, F_max,
                 alpha_critic=0.1):

        super().__init__(n_var=n_var, n_obj=n_obj, n_constr=0,
                         xl=xl, xu=xu, elementwise_evaluation=False)

        self.surrogate_pools = surrogate_pools
        self.D_net = D_net
        self.device = device
        self.X_min = X_min
        self.X_max = X_max
        self.F_min = F_min
        self.F_max = F_max
        self.alpha_critic = alpha_critic

    def _evaluate(self, X, out, *args, **kwargs):
        F_crit = critical_fitness(
            X,
            self.surrogate_pools,
            self.D_net,
            self.device,
            self.X_min, self.X_max, self.F_min, self.F_max,
            alpha_critic=self.alpha_critic
        )
        out["F"] = F_crit

# Main

### 1. Initial settings

In [5]:
# Problem: dtlz1-7, omnitest
problem_name = 'omnitest'

if 'dtlz' in problem_name:
  n_var = 10
  n_obj = 2
  problem = get_problem(problem_name, n_var=n_var, n_obj=n_obj)

elif 'omnitest' in problem_name:
  n_var = 2
  n_obj = 2
  problem = OmniTest(n_var=n_var)

#sample_size = 11*nvars-1
sample_size = 1000

sampling = LHS()
X_init = sampling(problem, sample_size,seed=42).get("X")
F_init = problem.evaluate(X_init, return_values_of=["F"])

print('problem_name', problem_name)
print('X_init', X_init.shape)
print('F_init', F_init.shape)

# Metrics: HV
if problem_name == 'dtlz1':
  obj_min = np.array([0,0])
  obj_max = np.array([552.30,568.36])

if problem_name == 'dtlz2':
  obj_min = np.array([0,0])
  obj_max = np.array([2.78,2.93])

if problem_name == 'dtlz3':
  obj_min = np.array([0,0])
  obj_max = np.array([1605.54,1670.48])

if problem_name == 'dtlz4':
  obj_min = np.array([0,0])
  obj_max = np.array([2.83,2.78])

if problem_name == 'dtlz5':
  obj_min = np.array([0,0])
  obj_max = np.array([2.61,2.70])

if problem_name == 'dtlz6':
  obj_min = np.array([0,0])
  obj_max = np.array([9.78,9.78])

if problem_name == 'dtlz7':
  obj_min = np.array([0,0])
  obj_max = np.array([1.10,33.43])

if problem_name == 'omnitest':
  obj_min = np.array([-2,-2])
  obj_max = np.array([2.40,2.40])

ref_point = np.array([1.1,1.1])
hv = HV(ref_point=ref_point)
print('\nMin-Max normalization -> Min: ', obj_min)
print('Min-Max normalization -> Max: ', obj_max)
print('HV Reference points: ', ref_point)

problem_name omnitest
X_init (1000, 2)
F_init (1000, 2)

Min-Max normalization -> Min:  [-2 -2]
Min-Max normalization -> Max:  [2.4 2.4]
HV Reference points:  [1.1 1.1]


In [6]:
X_init

array([[1.47253591, 3.60173333],
       [5.07193713, 5.97454806],
       [1.53576486, 3.19300391],
       ...,
       [0.05110892, 2.51653149],
       [5.74226104, 5.96195231],
       [0.8895074 , 3.8673152 ]])

### 2. Surrogates

In [7]:
X_min = X_init.min(axis=0)
X_max = X_init.max(axis=0)
F_min = F_init.min(axis=0)
F_max = F_init.max(axis=0)

Xn = 2.0 * (X_init - X_min) / (X_max - X_min + 1e-12) - 1.0
Fn = 2.0 * (F_init - F_min) / (F_max - F_min + 1e-12) - 1.0
joint_init = np.hstack([Xn, Fn])
D_dim = joint_init.shape[1]

G, D_net, device = train_wgan_gp(
    joint_init=joint_init,
    D_dim=D_dim,
    n_epochs=2000,
    batch_size=64,
    z_dim=32,
    lambda_gp=10.0,
    n_critic=5,
    lr=1e-4,
    verbose=True
)

Training WGAN-GP...
Epoch 500/2000 | D_loss: 9.2278 | G_loss: 0.6950
Epoch 1000/2000 | D_loss: 8.2382 | G_loss: 0.6649
Epoch 1500/2000 | D_loss: 7.8620 | G_loss: 0.6290
Epoch 2000/2000 | D_loss: 10.0001 | G_loss: 0.7008
WGAN-GP training done.



In [8]:
K_models = n_var
surrogate_pools, norm_bounds = construct_surrogate_pool_with_GAN(
    X_init=X_init,
    F_init=F_init,
    G=G,
    D_net=D_net,
    device=device,
    n_models=K_models,
    select_ratio=0.2,
    poly_degree=2,
    gamma_rbfn=0.5,
    lambda_rbfn=1e-6
)

X_min, X_max, F_min, F_max = norm_bounds

dd_problem = DDMOEA_GAN_Problem(
    n_var=n_var,
    n_obj=n_obj,
    xl=problem.xl,
    xu=problem.xu,
    surrogate_pools=surrogate_pools,
    D_net=D_net,
    device=device,
    X_min=X_min,
    X_max=X_max,
    F_min=F_min,
    F_max=F_max,
    alpha_critic=0.1
)

Building surrogate pool with GAN augmentation ...
  Surrogate model 1/2 built. Train size = 1200
  Surrogate model 2/2 built. Train size = 1200
Surrogate pool ready.



### 3. Optimization

In [9]:
algorithm = NSGA2(
    pop_size=100,
    crossover=SBX(prob=1.0, eta=20),
    mutation=PM(prob=1.0 / n_var, eta=20),
    eliminate_duplicates=True)

mse_list, igd_list, hv_surrogate_list, hv_real_list,  = [], [], [], []

n_points = 200
if problem_name == 'dtlz5':
    X_opt = np.full((n_points, n_var), 0.5)
    X_opt[:, 0] = np.linspace(0, 1, n_points)
    pf = problem.evaluate(X_opt)

elif problem_name == 'dtlz6':
    X_opt = np.zeros((n_points, n_var))
    X_opt[:, 0] = np.linspace(0, 1, n_points)
    pf = problem.evaluate(X_opt)

elif problem_name == 'dtlz7':
    X_opt = np.zeros((n_points, n_var))
    X_opt[:, :n_obj-1] = np.linspace(0, 1, n_points).reshape(-1, 1)
    pf = problem.evaluate(X_opt)
else:
    pf = problem.pareto_front()
igd_plus = IGDPlus(pf)

for seed in range(1, 31):

    start_time = time.time()

    res = minimize(
        dd_problem,
        algorithm,
        ("n_gen", 100),
        seed=seed,
        verbose=False,
        save_history=True)

    end_time = time.time()

    solution = res.history[-1].opt.get("X")
    obj = res.history[-1].opt.get("F")
    f_real = problem.evaluate(solution, return_values_of=["F"])

    # MSE
    mse = mean_squared_error(f_real, obj)
    mse_list.append(mse)
    # IGD+
    igd_plus_real = float(igd_plus(f_real))
    igd_list.append(igd_plus_real)
    # HV
    f_real_normalization = (f_real - obj_min) / (obj_max - obj_min)
    obj_normalization = (obj - obj_min) / (obj_max - obj_min)
    hv_real = float(hv.do(f_real_normalization))
    hv_surrogate = float(hv.do(obj_normalization))
    hv_real_list.append(hv_real)
    hv_surrogate_list.append(hv_surrogate)


    max_obj = np.max(obj, axis=0)
    max_obj_real = np.max(f_real, axis=0)
    print(f"\nSeed {seed} | Time: {end_time - start_time:.2f}s | "
          f"MSE: {mse:.2e} | "
          f"IGD+: {igd_plus_real:.2e} | "
          f"Sur HV: {hv_surrogate:.2f} | "
          f"Real HV: {hv_real:.2f}")
    print(f"Max f_DDMOEA-GAN: {max_obj} | Max f_real: {max_obj_real}")


Seed 1 | Time: 1.72s | MSE: 1.32e+00 | IGD+: 3.24e+00 | Sur HV: 0.44 | Real HV: 0.19
Max f_DDMOEA-GAN: [-0.06979159 -0.05564973] | Max f_real: [1.57359949 0.16083783]

Seed 2 | Time: 1.74s | MSE: 1.32e+00 | IGD+: 3.24e+00 | Sur HV: 0.44 | Real HV: 0.19
Max f_DDMOEA-GAN: [-0.06979157 -0.05564973] | Max f_real: [1.57359334 0.16223706]

Seed 3 | Time: 2.26s | MSE: 1.32e+00 | IGD+: 3.24e+00 | Sur HV: 0.44 | Real HV: 0.19
Max f_DDMOEA-GAN: [-0.06979159 -0.05564971] | Max f_real: [1.57363789 0.16083943]

Seed 4 | Time: 1.49s | MSE: 1.32e+00 | IGD+: 3.24e+00 | Sur HV: 0.44 | Real HV: 0.19
Max f_DDMOEA-GAN: [-0.06979156 -0.05564989] | Max f_real: [1.57297182 0.16130123]

Seed 5 | Time: 1.72s | MSE: 1.32e+00 | IGD+: 3.24e+00 | Sur HV: 0.44 | Real HV: 0.19
Max f_DDMOEA-GAN: [-0.06979154 -0.05564974] | Max f_real: [1.57357682 0.1611948 ]

Seed 6 | Time: 1.74s | MSE: 1.32e+00 | IGD+: 3.24e+00 | Sur HV: 0.44 | Real HV: 0.19
Max f_DDMOEA-GAN: [-0.06979158 -0.05564975] | Max f_real: [1.57355854 0.16

In [10]:
mean_mse, std_mse = mean_std(mse_list)
mean_igd, std_igd = mean_std(igd_list)
mean_hv_real, std_hv_real = mean_std(hv_real_list)
mean_hv_surrogate, std_hv_surrogate = mean_std(hv_surrogate_list)

print('Problem name: ', problem_name)
print("\n=== DDMOEA-GAN: Statistics over 30 runs ===")

print(f"MSE: Mean = {mean_mse:.2e}, Std = {std_mse:.2e}")
print(f"IGD+: Mean = {mean_igd:.2e}, Std = {std_igd:.2e}")
print(f"Sur HV: Mean = {mean_hv_surrogate:.2f}, Std = {std_hv_surrogate:.2f}")
print(f"Real HV: Mean = {mean_hv_real:.2f}, Std = {std_hv_real:.2f}")

Problem name:  omnitest

=== DDMOEA-GAN: Statistics over 30 runs ===
MSE: Mean = 1.32e+00, Std = 1.19e-03
IGD+: Mean = 3.24e+00, Std = 1.22e-04
Sur HV: Mean = 0.44, Std = 0.00
Real HV: Mean = 0.19, Std = 0.00


In [11]:
print(problem_name+'_MSE_DDMOEA/GAN'+' =',mse_list)
print(problem_name+'_IGD+_DDMOEA/GAN'+' =',igd_list)
print(problem_name+'_HV_DDMOEA/GAN'+' =',hv_real_list)

omnitest_MSE_DDMOEA/GAN = [1.3244852889180205, 1.3202353437410843, 1.3237052599526649, 1.3230581680165985, 1.3233254312222784, 1.321867725487528, 1.3235201122027653, 1.3239560522511427, 1.32322210711258, 1.3217445475884095, 1.3215084332432914, 1.3217902981145726, 1.3231991868620079, 1.322428278539084, 1.3246142392644296, 1.3230828912799486, 1.3216607465075654, 1.3254209182412493, 1.3226950814967395, 1.3222513178603283, 1.322010383280444, 1.3226704179156816, 1.3228952175020126, 1.32231469584279, 1.3212569526758988, 1.3218276566723692, 1.3231779052604264, 1.3202012383903248, 1.323146176031432, 1.3242718295188127]
omnitest_IGD+_DDMOEA/GAN = [3.2384733773362226, 3.2382331332438317, 3.238511287865811, 3.238455212822633, 3.2384484330707908, 3.238470647955469, 3.2384074423686617, 3.2384883531458524, 3.2384110204489702, 3.2384420811898185, 3.238867243988241, 3.238360277527604, 3.238451603805505, 3.2384512341922767, 3.238462411151405, 3.238435262484187, 3.2385557119683406, 3.2384821899839373, 3

In [12]:
plot_obj_2d(obj, xlim=(-100, 600), ylim=(-100, 600))
plot_obj_2d(f_real, xlim=(-100, 600), ylim=(-100, 600))
plot_obj_2d(f_real_normalization, xlim=(-0.1, 1.5), ylim=(-0.1, 1.5))